# Implementation of Generative Forecasting using Joint Probability Models on KS PDE

In [15]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg
import sys
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
import torch.optim as optim

NOTEBOOK_NAME = "KS_Equation_Distribution_Plot_Conditional_NADE.ipynb"
NOTEBOOK_DIR_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "02_Conditional_NADE" / "Distrivution_Adherence_test",
    Path.cwd() / "Distrivution_Adherence_test",
]

for candidate_dir in NOTEBOOK_DIR_CANDIDATES:
    if (candidate_dir / NOTEBOOK_NAME).exists():
        NOTEBOOK_OUTPUT_DIR = candidate_dir
        break
else:
    NOTEBOOK_OUTPUT_DIR = Path.cwd()

OUTPUT_LOG_PATH = NOTEBOOK_OUTPUT_DIR / "conditional_nade_outputs.txt"
OUTPUT_LOG_PATH.write_text("Conditional NADE notebook outputs\n", encoding="utf-8")

_plot_counter = 0

def log_output(*values, sep=" ", end="\n"):
    text = sep.join(str(value) for value in values) + end
    with OUTPUT_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(text)

def save_current_figure(name, dpi=300):
    global _plot_counter
    _plot_counter += 1
    safe_name = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in name).strip("_")
    figure_path = NOTEBOOK_OUTPUT_DIR / f"{_plot_counter:02d}_{safe_name}.png"
    plt.savefig(figure_path, dpi=dpi, bbox_inches="tight")
    plt.close()
    log_output(f"Saved figure: {figure_path}")

log_output(f"Notebook output directory: {NOTEBOOK_OUTPUT_DIR}")


In [16]:
DATASET_CANDIDATES = [
    Path(r"C:\Users\arred\Documents\MTP\ks_dataset_imex.npy"),
    Path(r"C:\Users\arred\Documents\MTP\KS_Dataset_IMEX.npy"),
    NOTEBOOK_OUTPUT_DIR / "ks_dataset_imex.npy",
    NOTEBOOK_OUTPUT_DIR.parent / "ks_dataset_imex.npy",
    NOTEBOOK_OUTPUT_DIR / "02_Conditional_NADE" / "ks_dataset_imex.npy",
    NOTEBOOK_OUTPUT_DIR.parent / "02_Conditional_NADE" / "ks_dataset_imex.npy",
]

for dataset_path in DATASET_CANDIDATES:
    if dataset_path.exists():
        break
else:
    raise FileNotFoundError("Could not find ks_dataset_imex.npy. Checked C:\\Users\\arred\\Documents\\MTP and project-local folders.")

dataset = np.load(dataset_path)
log_output(f"Loaded dataset from: {dataset_path}")
log_output(dataset.shape)


In [17]:
data_slice=dataset[int(1e6):int(1e6)+1000:]
plot_data=data_slice.T

dt=0.1
x_min,x_max=-25,25

t_start = 0
t_end = data_slice.shape[0] * dt 

plt.figure(figsize=(10,6))
plt.imshow(plot_data,
           aspect='auto',
           origin='lower',
           extent=[t_start,t_end,x_min,x_max],
           cmap='viridis',
           vmin=-2.88,
           vmax=2.88)

plt.title("Ground Truth")
plt.xlabel("t (time units)")
plt.ylabel("x (distance along the domain)")
plt.xticks([0, 20, 40, 60, 80, 100])
plt.tight_layout()
plt.colorbar()
save_current_figure("ground_truth")


In [18]:
dataset.shape

(4000000, 200)

In [19]:
class ConditionalWindowDataset(Dataset):
    def __init__(self, trajectory, history_len=2, target_len=2, num_windows=1_000_000):
        self.trajectory = trajectory
        self.history_len = history_len
        self.target_len = target_len
        self.num_windows = min(num_windows, len(trajectory) - history_len - target_len + 1)

        data_for_stats = trajectory[:self.num_windows + history_len + target_len - 1]
        mean_state = data_for_stats.mean(axis=0).astype(np.float32)
        std_state = data_for_stats.std(axis=0).astype(np.float32) + 1e-8

        self.mean = np.tile(mean_state, history_len).astype(np.float32)
        self.std = np.tile(std_state, history_len).astype(np.float32)
        self.state_mean = mean_state
        self.state_std = std_state

    def __len__(self):
        return self.num_windows

    def __getitem__(self, idx):
        history = self.trajectory[idx:idx + self.history_len]
        target = self.trajectory[idx + 1:idx + 1 + self.target_len]

        history = history.reshape(-1).astype(np.float32)
        target = target.reshape(-1).astype(np.float32)

        history = (history - self.mean) / self.std
        target = (target - self.mean) / self.std

        return torch.from_numpy(history), torch.from_numpy(target)

    def normalize_window(self, window):
        return (window.reshape(-1).astype(np.float32) - self.mean) / self.std

    def unnormalize_window(self, window):
        return (window * self.std) + self.mean


In [20]:
train_dataset = ConditionalWindowDataset(dataset, history_len=2, target_len=2, num_windows=1_000_000)


In [21]:
train_dataloader = DataLoader(train_dataset, batch_size=2000, shuffle=True, num_workers=0)


In [22]:
log_output(f'Number of time windows (samples) in the Training Dataset = {len(train_dataset)}')


In [23]:
class ConditionalNADE(nn.Module):
    def __init__(self, target_dim=400, history_dim=400, hidden_dim=1500):
        super().__init__()
        self.target_dim = target_dim
        self.history_dim = history_dim
        self.hidden_dim = hidden_dim

        self.W = nn.Parameter(torch.randn(hidden_dim, target_dim) * 0.01)
        self.c = nn.Parameter(torch.zeros(hidden_dim))
        self.U = nn.Linear(history_dim, hidden_dim, bias=False)

        self.V_mu = nn.Parameter(torch.randn(target_dim, hidden_dim) * 0.01)
        self.b_mu = nn.Parameter(torch.zeros(target_dim))

        self.V_log_sigma = nn.Parameter(torch.randn(target_dim, hidden_dim) * 0.01)
        self.b_log_sigma = nn.Parameter(torch.zeros(target_dim))

    def forward(self, x, history):
        B = x.shape[0]
        mu_out = torch.zeros(B, self.target_dim, device=x.device)
        log_sigma_out = torch.zeros(B, self.target_dim, device=x.device)
        a = self.c.unsqueeze(0).expand(B, -1) + self.U(history)

        for i in range(self.target_dim):
            h = torch.relu(a)
            mu_i = h @ self.V_mu[i] + self.b_mu[i]
            log_sigma_i = h @ self.V_log_sigma[i] + self.b_log_sigma[i]
            log_sigma_i = torch.clamp(log_sigma_i, min=-5, max=2)
            mu_out[:, i] = mu_i
            log_sigma_out[:, i] = log_sigma_i

            if i < self.target_dim - 1:
                v_i = x[:, i].unsqueeze(1)
                W_i = self.W[:, i].unsqueeze(0)
                a = a + v_i * W_i

        return mu_out, log_sigma_out

    @torch.no_grad()
    def sample(self, history):
        B = history.shape[0]
        samples = torch.zeros(B, self.target_dim, device=history.device)
        a = self.c.unsqueeze(0).expand(B, -1) + self.U(history)

        for i in range(self.target_dim):
            h = torch.relu(a)
            mu_i = h @ self.V_mu[i] + self.b_mu[i]
            log_sigma_i = h @ self.V_log_sigma[i] + self.b_log_sigma[i]
            log_sigma_i = torch.clamp(log_sigma_i, min=-5, max=2)
            sigma_i = torch.exp(log_sigma_i)
            x_i = mu_i + sigma_i * torch.randn(B, device=history.device)
            samples[:, i] = x_i

            if i < self.target_dim - 1:
                a = a + x_i.unsqueeze(1) * self.W[:, i].unsqueeze(0)

        return samples


In [24]:
test_idx = int(1e6)
initial_history = dataset[test_idx:test_idx + 2].astype(np.float32)
log_output(initial_history.shape)


In [25]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
CHECKPOINT_CANDIDATES = [
    NOTEBOOK_OUTPUT_DIR / "Conditional_Nade.pt",
    NOTEBOOK_OUTPUT_DIR.parent / "Conditional_Nade.pt",
    NOTEBOOK_OUTPUT_DIR / "02_Conditional_NADE" / "Conditional_Nade.pt",
    NOTEBOOK_OUTPUT_DIR.parent / "02_Conditional_NADE" / "Conditional_Nade.pt",
]

for checkpoint_path in CHECKPOINT_CANDIDATES:
    if checkpoint_path.exists():
        break
else:
    raise FileNotFoundError("Could not find Conditional_Nade.pt. Update CHECKPOINT_CANDIDATES with the checkpoint location.")

checkpoint = torch.load(checkpoint_path, map_location=device)
model = ConditionalNADE(target_dim=400, history_dim=400, hidden_dim=1500).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

log_output(f"Loaded conditional NADE from: {checkpoint_path}")
log_output(f"Checkpoint epoch: {checkpoint.get('epoch')} | best loss: {checkpoint.get('best_loss'):.4f}")


The conditional rollout follows the Lorenz conditional NADE implementation: sample many candidate two-state windows conditioned on the current two-state history, match the first predicted state to the most recent state, and use the winning second state as the forecast.


In [26]:
N = 100000
forecast_horizon = int(4e4)
forecasted_trajectory = []

current_history_unnorm = torch.tensor(initial_history.reshape(-1), dtype=torch.float32, device=device)
mu_tensor = torch.tensor(train_dataset.mean, dtype=torch.float32, device=device)
std_tensor = torch.tensor(train_dataset.std, dtype=torch.float32, device=device)

base_model = model.module if isinstance(model, nn.DataParallel) else model
base_model.eval()

with torch.no_grad():
    for t in range(forecast_horizon):
        current_history_norm = (current_history_unnorm - mu_tensor) / (std_tensor + 1e-8)
        batch_history = current_history_norm.unsqueeze(0).expand(N, -1)

        candidates_norm = base_model.sample(batch_history)
        candidates_unnorm = (candidates_norm * std_tensor) + mu_tensor

        candidate_tails = candidates_unnorm[:, :200]
        candidate_heads = candidates_unnorm[:, 200:]
        most_recent_observation = current_history_unnorm[200:]

        distances = torch.norm(candidate_tails - most_recent_observation, dim=1)
        best_index = torch.argmin(distances)
        winning_forecast = candidate_heads[best_index]

        forecasted_trajectory.append(winning_forecast.cpu().numpy())
        current_history_unnorm = torch.cat([most_recent_observation, winning_forecast])

        if (t + 1) % 100 == 0:
            log_output(f"Forecasted {t + 1}/{forecast_horizon} steps")

forecasted_trajectory = np.asarray(forecasted_trajectory, dtype=np.float32)
log_output(f"Forecasting complete! Shape: {forecasted_trajectory.shape}")


KeyboardInterrupt: 

In [ ]:
forecast_path = NOTEBOOK_OUTPUT_DIR / "conditional_nade_forecasted_trajectory.npy"
np.save(forecast_path, forecasted_trajectory)
log_output(f"Saved forecasted trajectory: {forecast_path}")


In [ ]:
truth_data = dataset[test_idx + 2:test_idx + 2 + forecast_horizon].astype(np.float32)
mae_by_step = np.mean(np.abs(forecasted_trajectory[:len(truth_data)] - truth_data), axis=1)
cumulative_mae = np.cumsum(mae_by_step) / np.arange(1, len(mae_by_step) + 1)

mae_path = NOTEBOOK_OUTPUT_DIR / "conditional_nade_mae_by_step.txt"
np.savetxt(mae_path, mae_by_step, header="MAE per forecast step")
log_output(f"Saved MAE values: {mae_path}")
log_output(f"Mean MAE over horizon: {mae_by_step.mean():.6f}")
log_output(f"Median MAE over horizon: {np.median(mae_by_step):.6f}")
log_output(f"Final cumulative MAE: {cumulative_mae[-1]:.6f}")


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(mae_by_step, linewidth=1.0, label="Step MAE")
plt.plot(cumulative_mae, linewidth=2.0, label="Cumulative mean MAE")
plt.xlabel("Forecast step")
plt.ylabel("MAE")
plt.title("Conditional NADE Forecast MAE")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
save_current_figure("conditional_nade_forecast_mae")


In [ ]:
mae_windows = [100, 500, 1000, 5000, 10000, forecast_horizon]
mae_summary = []
for window in mae_windows:
    usable = min(window, len(mae_by_step))
    mae_summary.append((usable, mae_by_step[:usable].mean()))

mae_summary_path = NOTEBOOK_OUTPUT_DIR / "conditional_nade_mae_summary.txt"
with mae_summary_path.open("w", encoding="utf-8") as f:
    f.write("steps,mean_mae\n")
    for usable, value in mae_summary:
        f.write(f"{usable},{value:.6f}\n")
        log_output(f"MAE over first {usable:>6} steps: {value:.6f}")
log_output(f"Saved MAE summary: {mae_summary_path}")


In [ ]:
log_output(f"Forecasting complete! Shape: {forecasted_trajectory.shape}")

In [ ]:
data_slice=forecasted_trajectory[:1000,:]
plot_data=data_slice.T

dt=0.1
x_min,x_max=-25,25

t_start = 0
t_end = data_slice.shape[0] * dt 

plt.figure(figsize=(10,6))
plt.imshow(plot_data,
           aspect='auto',
           origin='lower',
           extent=[t_start,t_end,x_min,x_max],
           cmap='viridis',
           vmin=-2.88,
           vmax=2.88)

plt.title("Forecasted using Generative Model: Conditional NADE")
plt.xlabel("t (time units)")
plt.ylabel("x (distance along the domain)")
plt.xticks([0, 20, 40, 60, 80, 100])
plt.tight_layout()
plt.colorbar()
save_current_figure("conditional_nade_forecast_heatmap")


In [ ]:
sample_step = min(100, len(forecasted_trajectory) - 1)

plt.figure(figsize=(10, 3))
plt.plot(truth_data[sample_step], label="truth")
plt.plot(forecasted_trajectory[sample_step], label="conditional NADE forecast")
plt.xlabel("Spatial index")
plt.ylabel("State value")
plt.title(f"Forecast sample at step {sample_step}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
save_current_figure("conditional_nade_truth_vs_forecast_sample")


In [ ]:
# real one-step change over the evaluation segment
real_deltas = np.linalg.norm(truth_data[1:] - truth_data[:-1], axis=1)

# generated one-step change over the conditional forecast
pred_deltas = np.linalg.norm(forecasted_trajectory[1:] - forecasted_trajectory[:-1], axis=1)

delta_stats_path = NOTEBOOK_OUTPUT_DIR / "conditional_nade_delta_stats.txt"
with delta_stats_path.open("w", encoding="utf-8") as f:
    f.write(f"Real delta mean: {real_deltas.mean()}\n")
    f.write(f"Generated delta mean: {pred_deltas.mean()}\n")
    f.write(f"Real delta std: {real_deltas.std()}\n")
    f.write(f"Generated delta std: {pred_deltas.std()}\n")

log_output(f"Real delta mean: {real_deltas.mean()}")
log_output(f"Generated delta mean: {pred_deltas.mean()}")
log_output(f"Real delta std: {real_deltas.std()}")
log_output(f"Generated delta std: {pred_deltas.std()}")
log_output(f"Saved delta stats: {delta_stats_path}")


# Distribution Adherence Plots

In [ ]:
idx = int(1e6)

all_data = dataset[:]
train_data = dataset[:idx]
truth_data = dataset[test_idx + 2:test_idx + 2 + forecast_horizon]
pred_data = forecasted_trajectory[:len(truth_data)]


In [ ]:
all_vals = np.asarray(all_data).reshape(-1)
train_vals = np.asarray(train_data).reshape(-1)
truth_vals = np.asarray(truth_data).reshape(-1)
pred_vals = np.asarray(pred_data).reshape(-1)

bins = np.linspace(
    min(all_vals.min(), train_vals.min(), truth_vals.min(), pred_vals.min()),
    max(all_vals.max(), train_vals.max(), truth_vals.max(), pred_vals.max()),
    80
)

eps = 1e-12

all_hist, edges = np.histogram(all_vals, bins=bins, density=True)
train_hist, _ = np.histogram(train_vals, bins=bins, density=True)
truth_hist, _ = np.histogram(truth_vals, bins=bins, density=True)
pred_hist, _ = np.histogram(pred_vals, bins=bins, density=True)

centers = 0.5 * (edges[:-1] + edges[1:])

all_hist = all_hist + eps
train_hist = train_hist + eps
truth_hist = truth_hist + eps
pred_hist = pred_hist + eps

distribution_histograms_path = NOTEBOOK_OUTPUT_DIR / "conditional_nade_distribution_histograms.txt"
np.savetxt(
    distribution_histograms_path,
    np.column_stack([centers, all_hist, train_hist, truth_hist, pred_hist]),
    header="center all_density train_density truth_density prediction_density"
)
log_output(f"Saved distribution histogram values: {distribution_histograms_path}")

left_limit = np.percentile(all_vals, 5)
right_limit = np.percentile(all_vals, 95)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Main plot
ax = axes[0]
ax.fill_between(centers, all_hist, alpha=0.35, label="All data")
ax.plot(centers, train_hist, color="black", linewidth=1.5, label="Training set")
ax.plot(centers, truth_hist, color="steelblue", linewidth=1.5, label="Truth")
ax.plot(centers, pred_hist, color="crimson", linestyle="--", linewidth=1.5, label="Prediction")
ax.set_yscale("log")
ax.set_xlabel("State value")
ax.set_ylabel("Probability density")
ax.set_title("Main Distribution")
ax.legend()

# Left tail
ax = axes[1]
ax.fill_between(centers, all_hist, alpha=0.35)
ax.plot(centers, train_hist, color="black", linewidth=1.5)
ax.plot(centers, truth_hist, color="steelblue", linewidth=1.5)
ax.plot(centers, pred_hist, color="crimson", linestyle="--", linewidth=1.5)
ax.set_yscale("log")
ax.set_xlim(centers.min(), left_limit)
ax.set_xlabel("State value")
ax.set_ylabel("Probability density")
ax.set_title("Left Tail View")

# Right tail
ax = axes[2]
ax.fill_between(centers, all_hist, alpha=0.35)
ax.plot(centers, train_hist, color="black", linewidth=1.5)
ax.plot(centers, truth_hist, color="steelblue", linewidth=1.5)
ax.plot(centers, pred_hist, color="crimson", linestyle="--", linewidth=1.5)
ax.set_yscale("log")
ax.set_xlim(right_limit, centers.max())
ax.set_xlabel("State value")
ax.set_ylabel("Probability density")
ax.set_title("Right Tail View")

plt.tight_layout()
save_current_figure("conditional_nade_distribution_adherence")
